## Clean Text

In [ ]:
import pandas as pd
import re
from pathlib import Path

# ========= Path =========
input_path = Path("dataset/alltext.xlsx")
output_path = Path("dataset/alltext_cleaned.csv")

# ========= data =========
df = pd.read_excel(input_path)

# ========= Keep only the "text" column =========
df = df[["text"]].copy()

# ========= Clean text =========
def clean_text(text):
    if pd.isna(text):
        return None

    text = str(text)

    # 去掉多余换行、回车、制表符
    text = re.sub(r"[\r\n\t]+", " ", text)

    # 去掉特殊符号（保留字母、数字、基本标点）
    text = re.sub(r"[^\w\s,.!?;:'\"()\-/%]", " ", text)

    # 去掉重复空格
    text = re.sub(r"\s+", " ", text).strip()

    # 清洗后为空则记为缺失值
    return text if text != "" else None

# ========= Clean =========
df["text"] = df["text"].apply(clean_text)

# ========= Remove null values =========
df = df.dropna(subset=["text"]).reset_index(drop=True)

# ========= Save as CSV =========
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("清洗完成，文件已保存到：", output_path)
print("保留行数：", len(df))

清洗完成，文件已保存到： dataset/alltext_cleaned.csv
保留行数： 330


## Triple

In [ ]:
from openai import OpenAI
import pandas as pd
import json
import time
from tqdm import tqdm
from getpass import getpass

client = OpenAI(api_key=" ",
                base_url="https://")

def get_triplets_from_ai(chunk_text):
    """Call API to extract triplets in English"""
    system_prompt = """
    You are a board-certified endocrinologist and professional medical fact-checker specializing in diabetes care. Your core responsibility is to verify the accuracy of diabetes-related information from social media short videos, extract standardized medical triples (Subject - Predicate - Object), and ensure consistency with authoritative medical knowledge graph.
    Requirements:
    1. Format: MUST be a standard JSON list of objects: [{"S": "Subject", "P": "Predicate", "O": "Object"}].
    2. Predicate (P) should be professional and standardized, such as: 
    [treats, causes, symptom_of, prevents, belongs_to, contraindicated_for, increases_risk_of, lowers_blood_glucose, associated_with].
    3. Language: The output MUST be in English.
    4. If no clear triplets are found, return an empty list [].
    """

    # Few-shot Example in English
    user_example = "Text: 'Metformin can effectively lower blood glucose levels in patients with Type 2 Diabetes.'\nOutput:"
    assistant_example = '[{"S": "Metformin", "P": "lowers_blood_glucose", "O": "Type 2 Diabetes"}]'

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_example},
                {"role": "assistant", "content": assistant_example},
                {"role": "user", "content": f"Text to process: '{chunk_text}'\nOutput:"}
            ],
            response_format={ "type": "json_object" }
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error during API call: {e}")
        return "[]"

In [9]:
df = pd.read_csv("dataset/alltext_cleaned.csv", encoding="utf-8")

# 检查 'text' 列是否存在
if "text" not in df.columns:
    raise ValueError("The column named 'text' was not found in the CSV file. Please check the column name.")

print(f"A total of {len(df)} pieces of text were read.")

A total of 330 pieces of text were read.


In [10]:
triplets_list = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting triplets"):
    text = row["text"]
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        triplets_list.append("[]")  # 空文本返回空列表
        continue
    
    result_str = get_triplets_from_ai(text)
    # 尝试解析一下，确保返回的是合法 JSON（可选）
    try:
        # 验证是否能解析为 JSON，如果失败则存储空列表
        json.loads(result_str)
        triplets_list.append(result_str)
    except json.JSONDecodeError:
        print(f"Warning: The {idx}th item returned is not a valid JSON and has been emptied. Content: {result_str[:100]}")
        triplets_list.append("[]")
    
    time.sleep(0.5)  # 控制请求频率

df["triplets_json"] = triplets_list

Extracting triplets: 100%|██████████| 330/330 [45:55<00:00,  8.35s/it]


In [ ]:
output_file = "output_data/text_with_triplets_ver1.json"
df.to_json(output_file, orient="records", force_ascii=False, indent=2)
print(f"The result has been saved to {output_file}")

The result has been saved to output_data/text_with_triplets.json


In [ ]:
expanded_rows = []
for idx, row in df.iterrows():
    try:
        triplets = json.loads(row["triplets_json"])
        for t in triplets:
            expanded_rows.append({
                "source_row": idx,
                "subject": t.get("S", ""),
                "predicate": t.get("P", ""),
                "object": t.get("O", "")
            })
    except:
        # 如果某行 JSON 解析失败，跳过
        continue

expanded_df = pd.DataFrame(expanded_rows)
# expanded_df.to_csv("expanded_triplets.csv", index=False, encoding="utf-8")
expanded_df.to_json(
    "output_data/expanded_triplets_ver1.json",
    orient="records",
    force_ascii=False,
    indent=2
)
print("The expanded triples have been saved.")

The expanded triples have been saved.


## Clean Triples

In [1]:
import pandas as pd
import re

# 输入输出路径
input_path = "output_data/expanded_triplets_ver1.json"
output_path = "output_data/expanded_triplets_ver1_clean.json"

# 读取数据
df = pd.read_json(input_path)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 小写
    text = text.lower()
    
    # 下划线 → 空格
    text = text.replace("_", " ")
    
    # 去多余空格
    text = re.sub(r"\s+", " ", text)
    
    # 去首尾标点
    text = text.strip(".,;:!? ")
    
    return text

# 应用清洗（不改变结构）
df["subject"] = df["subject"].apply(clean_text)
df["predicate"] = df["predicate"].apply(clean_text)
df["object"] = df["object"].apply(clean_text)

# 输出
df.to_json(
    output_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print("Cleaned triples saved (no deduplication).")
print("Total rows:", len(df))
print(df.head())

Cleaned triples saved (no deduplication).
Total rows: 2704
   source_row         subject   predicate               object
0           0  lantus insulin      treats      type 1 diabetes
1           0  lantus insulin      treats      type 2 diabetes
2           0  lantus insulin  belongs to  long-acting insulin
3           0  lantus insulin      causes      low blood sugar
4           0  lantus insulin      causes          weight gain


In [ ]:
# import json
# import re
# from pathlib import Path

# # ========= 1. File paths =========
# input_file = Path("output_data/expanded_triplets.json")
# output_file = Path("output_data/expanded_triplets_cleaned.json")


# # ========= 2. Text cleaning function =========
# def clean_text(text, lowercase=True):
#     """
#     基础文本清洗：
#     - 转为字符串
#     - 去掉首尾空格
#     - 合并多余空格/换行
#     - 统一常见引号、破折号
#     - 默认转小写
#     """
#     if text is None:
#         return ""

#     text = str(text)

#     # 统一换行和制表符为空格
#     text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")

#     # 统一常见引号
#     text = text.replace("“", '"').replace("”", '"')
#     text = text.replace("‘", "'").replace("’", "'")

#     # 统一破折号
#     text = text.replace("–", "-").replace("—", "-")

#     # 去掉多余空格
#     text = re.sub(r"\s+", " ", text).strip()

#     # 默认统一小写
#     if lowercase:
#         text = text.lower()

#     return text


# # ========= 3. Check empty triplet =========
# def is_empty_triplet(subj, pred, obj):
#     return subj == "" or pred == "" or obj == ""


# # ========= 4. Load JSON =========
# with open(input_file, "r", encoding="utf-8") as f:
#     data = json.load(f)

# if not isinstance(data, list):
#     raise ValueError("expanded_triplets.json 的最外层应为 list。")


# # ========= 5. Cleaning + Deduplication =========
# cleaned_data = []
# seen = set()

# for item in data:
#     if not isinstance(item, dict):
#         continue

#     subject_original = item.get("subject", "")
#     predicate_original = item.get("predicate", "")
#     object_original = item.get("object", "")

#     subject_clean = clean_text(subject_original, lowercase=True)
#     predicate_clean = clean_text(predicate_original, lowercase=True)
#     object_clean = clean_text(object_original, lowercase=True)

#     # 跳过空三元组
#     if is_empty_triplet(subject_clean, predicate_clean, object_clean):
#         continue

#     # 用 clean 版本做去重
#     triplet_key = (subject_clean, predicate_clean, object_clean)

#     if triplet_key in seen:
#         continue
#     seen.add(triplet_key)

#     new_item = item.copy()

#     # 保留原始字段，同时新增 clean 字段
#     new_item["subject_original"] = subject_original
#     new_item["predicate_original"] = predicate_original
#     new_item["object_original"] = object_original

#     new_item["subject_clean"] = subject_clean
#     new_item["predicate_clean"] = predicate_clean
#     new_item["object_clean"] = object_clean

#     cleaned_data.append(new_item)


# # ========= 6. Save result =========
# output_file.parent.mkdir(parents=True, exist_ok=True)

# with open(output_file, "w", encoding="utf-8") as f:
#     json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

# print(f"Cleaning completed. {len(cleaned_data)} triplets retained.")
# print(f"Saved to: {output_file}")

Cleaning completed. 2427 triplets retained.
Saved to: output_data/expanded_triplets_cleaned.json


In [ ]:
# # ========= 7. Only clean version =========
# simple_data = []

# for item in cleaned_data:
#     simple_data.append({
#         "subject": item["subject_clean"],
#         "predicate": item["predicate_clean"],
#         "object": item["object_clean"]
#     })

# simple_output_file = Path("output_data/expanded_triplets_cleaned_simple.json")

# with open(simple_output_file, "w", encoding="utf-8") as f:
#     json.dump(simple_data, f, ensure_ascii=False, indent=2)

# print(f"Simple version saved to: {simple_output_file}")

Simple version saved to: output_data/expanded_triplets_cleaned_simple.json
